In [ ]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import time

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

In [11]:
date = "current"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [12]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [13]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
39049,13896714,7940,ZAMBIA,DET,2025-11,13896714.0
39050,12624834,7950,ESWATINI,DET,2025-11,12624834.0
39051,15520151,7960,ZIMBABWE,DET,2025-11,15520151.0
39052,2368430,7970,MALAWI,DET,2025-11,2368430.0
39053,8270767,7990,LESOTHO,DET,2025-11,8270767.0


In [14]:
country_list[0] = ""

In [15]:
country_list.extend(["0003", "0020"])

In [16]:
len(country_list)

33

In [ ]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "from+2013-01" + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(xxx)
    
    url = surl + "&CTY_CODE=" + xxx
    
    if xxx == "":
        url = surl
    
    r = requests.get(url)

    while r.status_code != 200:
        print(f"Request failed with status {r.status_code}, waiting 30 seconds...")
        time.sleep(30)
        r = requests.get(url)
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)



Already have downloaded file
5700
<Response [500]>


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
foo.head()

NameError: name 'foo' is not defined

In [ ]:
foo.tail()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
1836155,USMCA (NAFTA),1793055,0,3907500000,ALKYD RESINS,2025-10,HS10,0020
1836156,USMCA (NAFTA),18225400,0,3907610010,POLYETHYLENE TEREPHTHALATE VISC GT=78 ML/G LT=88,2025-10,HS10,0020
1836157,USMCA (NAFTA),4799526,0,3907610050,POLYETHYLENE TEREPHTHALATE VISCSTY GT 88 ML/G,2025-10,HS10,0020
1836158,USMCA (NAFTA),2721085,0,3907690010,"OTHR POLYETHYLENE TEREPHTHALATE VIS GT=70ML/G,...",2025-10,HS10,0020
1836159,USMCA (NAFTA),4506380,4973,3907690050,OTHER POLYETHYLENE TEREPHTHALATE VISCOS GT= 78...,2025-10,HS10,0020
